# Báo cáo tổng quan dữ liệu Warehouse 3NF

Chạy cell Python để quét toàn bộ tệp CSV trong `warehouse_3nf` và tạo một báo cáo văn bản tổng hợp tại `warehouse_3nf/mo_ta_tong_quat.txt`.

Báo cáo gồm kích cỡ và số dòng/cột của từng bảng, cùng mô tả từng cột: kiểu dữ liệu suy luận, số lượng và tỷ lệ giá trị trống, số giá trị phân biệt, độ dài tối đa, min/max, trung bình (nếu là số) và một số giá trị mẫu.

In [ ]:
import csv
import json
import re
from datetime import date
from pathlib import Path


cwd = Path.cwd()
warehouse_candidates = [
    cwd / 'warehouse_3nf',
    cwd / 'DAAI_N1.4' / 'warehouse_3nf',
]
warehouse_dir = next((path for path in warehouse_candidates if path.is_dir()), None)
if warehouse_dir is None:
    raise FileNotFoundError(
        'Không tìm thấy warehouse_3nf. Hãy chạy notebook từ DAAI_N1.4 hoặc thư mục cha.'
    )

csv_paths = sorted(path for path in warehouse_dir.glob('*.csv') if path.is_file())
if not csv_paths:
    raise FileNotFoundError(f'Không tìm thấy tệp CSV nào trong {warehouse_dir}')

def detect_encoding(path):
    with path.open('rb') as file:
        prefix = file.read(4)
    if prefix[:2] in (bytes([255, 254]), bytes([254, 255])):
        return 'utf-16'
    return 'utf-8-sig'


def infer_value_type(value):
    if value.lower() in {'true', 'false'}:
        return 'boolean'
    if value.lstrip('+-').isdigit():
        return 'integer'
    try:
        float(value)
        return 'float'
    except ValueError:
        pass
    try:
        date.fromisoformat(value)
        return 'date'
    except ValueError:
        return 'string'


def summarize_column_type(types):
    if not types:
        return 'empty'
    if types <= {'integer', 'float'}:
        return 'integer' if types == {'integer'} else 'float'
    return next(iter(types)) if len(types) == 1 else 'mixed'


table_summary = []
column_profile = []

for csv_path in csv_paths:
    encoding = detect_encoding(csv_path)
    column_stats = None
    row_count = 0

    with csv_path.open('r', encoding=encoding, newline='') as file:
        reader = csv.reader(file)
        headers = next(reader, None)
        if not headers:
            raise ValueError(f'Tệp CSV rỗng hoặc thiếu header: {csv_path}')
        if len(headers) != len(set(headers)):
            raise ValueError(f'Tên cột bị trùng trong {csv_path.name}: {headers}')

        column_stats = {
            name: {
                'types': set(),
                'values': set(),
                'samples': [],
                'null_count': 0,
                'max_length': 0,
                'numeric_min': None,
                'numeric_max': None,
                'numeric_sum': 0.0,
                'numeric_count': 0,
                'date_min': None,
                'date_max': None,
            }
            for name in headers
        }

        for line_number, row in enumerate(reader, start=2):
            if len(row) != len(headers):
                raise ValueError(
                    f'Dòng CSV không đủ số trường tại {csv_path.name}:{line_number}'
                )
            row_count += 1
            for name, raw_value in zip(headers, row):
                stats = column_stats[name]
                if not raw_value.strip():
                    stats['null_count'] += 1
                    continue

                value = raw_value.strip()
                value_type = infer_value_type(value)
                stats['types'].add(value_type)
                stats['values'].add(value)
                stats['max_length'] = max(stats['max_length'], len(raw_value))
                if len(stats['samples']) < 3 and value not in stats['samples']:
                    stats['samples'].append(value)

                if value_type in {'integer', 'float'}:
                    number = float(value)
                    stats['numeric_min'] = (
                        number if stats['numeric_min'] is None
                        else min(stats['numeric_min'], number)
                    )
                    stats['numeric_max'] = (
                        number if stats['numeric_max'] is None
                        else max(stats['numeric_max'], number)
                    )
                    stats['numeric_sum'] += number
                    stats['numeric_count'] += 1
                elif value_type == 'date':
                    stats['date_min'] = (
                        value if stats['date_min'] is None
                        else min(stats['date_min'], value)
                    )
                    stats['date_max'] = (
                        value if stats['date_max'] is None
                        else max(stats['date_max'], value)
                    )

    table_summary.append({
        'table_name': csv_path.stem,
        'file_name': csv_path.name,
        'row_count': row_count,
        'column_count': len(headers),
        'file_size_bytes': csv_path.stat().st_size,
        'file_size_mb': round(csv_path.stat().st_size / (1024 ** 2), 3),
        'encoding': encoding,
    })

    for name in headers:
        stats = column_stats[name]
        non_null_count = row_count - stats['null_count']
        inferred_type = summarize_column_type(stats['types'])
        min_value = ''
        max_value = ''
        mean_value = ''
        if inferred_type in {'integer', 'float'} and stats['numeric_count']:
            min_value = stats['numeric_min']
            max_value = stats['numeric_max']
            mean_value = round(stats['numeric_sum'] / stats['numeric_count'], 6)
        elif inferred_type == 'date':
            min_value = stats['date_min']
            max_value = stats['date_max']

        column_profile.append({
            'table_name': csv_path.stem,
            'column_name': name,
            'inferred_type': inferred_type,
            'non_null_count': non_null_count,
            'null_count': stats['null_count'],
            'null_pct': round(stats['null_count'] / row_count * 100, 4) if row_count else 0,
            'distinct_count': len(stats['values']),
            'min_value': min_value,
            'max_value': max_value,
            'mean_value': mean_value,
            'max_length': stats['max_length'],
            'sample_values': json.dumps(stats['samples'], ensure_ascii=False),
        })


report_lines = [
    'BÁO CÁO TỔNG QUAN DỮ LIỆU - WAREHOUSE 3NF',
    '=' * 72,
    f'Thư mục dữ liệu: {warehouse_dir.resolve()}',
    f'Số bảng: {len(table_summary)}',
    f'Tổng số dòng: {sum(table["row_count"] for table in table_summary):,}',
    f'Tổng số cột: {sum(table["column_count"] for table in table_summary):,}',
    f'Tổng dung lượng CSV: {sum(table["file_size_bytes"] for table in table_summary):,} bytes',
    '',
    'CHI TIẾT THEO BẢNG',
    '=' * 72,
]

for table in table_summary:
    report_lines.extend([
        '',
        f'BẢNG: {table["table_name"]} ({table["file_name"]})',
        '-' * 72,
        f'Số dòng: {table["row_count"]:,} | Số cột: {table["column_count"]} | '
        f'Kích thước: {table["file_size_bytes"]:,} bytes ({table["file_size_mb"]:.3f} MB) | '
        f'Encoding: {table["encoding"]}',
        'Mô tả cột:',
    ])

    for column in (item for item in column_profile if item['table_name'] == table['table_name']):
        details = [
            f'type={column["inferred_type"]}',
            f'không trống={column["non_null_count"]:,}/{table["row_count"]:,}',
            f'trống={column["null_count"]:,} ({column["null_pct"]:.2f}%)',
            f'phân biệt={column["distinct_count"]:,}',
            f'dài tối đa={column["max_length"]} ký tự',
        ]
        if column['min_value'] != '':
            details.append(f'min={column["min_value"]}')
        if column['max_value'] != '':
            details.append(f'max={column["max_value"]}')
        if column['mean_value'] != '':
            details.append(f'trung bình={column["mean_value"]}')
        samples = json.loads(column['sample_values'])
        if samples:
            details.append(f'ví dụ={" | ".join(samples)}')
        report_lines.append(f'  - {column["column_name"]}: ' + '; '.join(details))

report_path = warehouse_dir / 'mo_ta_tong_quat.txt'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
report_text = '\n'.join(report_lines)
print(report_text)
print(f'\nĐã lưu báo cáo văn bản tại: {report_path.resolve()}')
